# American Option Pricing — NVIDIA GPU (Google Colab)

**Methods:**
- Monte Carlo European option (NumPy CPU vs CUDA GPU)
- PSO American option (NumPy CPU vs CUDA GPU: hybrid / scalar / fusion)
- Longstaff-Schwartz LSMC (NumPy CPU vs CUDA GPU)

> **Runtime:** Make sure you have selected **GPU** under *Runtime → Change runtime type → T4 GPU*

## 1. Setup

In [ ]:
# Install PyCUDA (only needed once per Colab session)
!pip install pycuda --quiet

In [ ]:
import subprocess, os

# Clone the repo if not already present
if not os.path.exists('/content/Fin_ParallelComputing'):
    subprocess.run(['git', 'clone',
                    'https://github.com/AIScorpio/Fin_ParallelComputing',
                    '/content/Fin_ParallelComputing'], check=True)

os.chdir('/content/Fin_ParallelComputing/src')
print('Working directory:', os.getcwd())

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
from models.utils import checkCUDA, CUDAEnv
checkCUDA()
print('Device:', CUDAEnv.deviceName)

## 2. Simulation Parameters

In [ ]:
# Option parameters
S0      = 100.0    # spot price
K       = 100.0    # strike price
r       = 0.03     # risk-free rate
sigma   = 0.30     # volatility
T       = 1.0      # maturity (years)
opttype = 'P'      # 'P' = Put, 'C' = Call

# Simulation parameters
nPath   = 65536    # Monte Carlo paths
nPeriod = 50       # time steps
nFish   = 500      # PSO particles

print(f'Paths={nPath:,}  Periods={nPeriod}  PSO particles={nFish}')

## 3. Monte Carlo — European Option

In [ ]:
from models.mc import hybridMonteCarlo

mc = hybridMonteCarlo(S0, r, sigma, T, nPath, nPeriod, K, opttype, nFish)

# CPU baseline
price_np, t_np = mc.getEuroOption_np()

# GPU – one-shot optimised (WINNER)
price_gpu, t_gpu = mc.getEuroOption_cl_optimized()

print(f'\nSpeedup: {t_np/t_gpu:.1f}x')

## 4. PSO — American Option

In [ ]:
from models.pso import PSO_Numpy, PSO_CUDA_hybrid, PSO_CUDA_scalar, PSO_CUDA_scalar_fusion

iterMax = 30

# CPU
pso_np = PSO_Numpy(mc, nFish, iterMax=iterMax)
price_pso_np, t_pso_np, *_ = pso_np.solvePsoAmerOption_np()

# GPU – hybrid
pso_hyb = PSO_CUDA_hybrid(mc, nFish, iterMax=iterMax)
price_hyb, t_hyb, *_ = pso_hyb.solvePsoAmerOption_cl()

# GPU – scalar (backward)
pso_sc = PSO_CUDA_scalar(mc, nFish, direction='backward', iterMax=iterMax)
price_sc, t_sc, *_ = pso_sc.solvePsoAmerOption_cl()

# GPU – fused
pso_fus = PSO_CUDA_scalar_fusion(mc, nFish, iterMax=iterMax)
price_fus, t_fus, *_ = pso_fus.solvePsoAmerOption_cl()

print(f'\nSpeedup (hybrid) : {t_pso_np/t_hyb:.1f}x')
print(f'Speedup (scalar) : {t_pso_np/t_sc:.1f}x')
print(f'Speedup (fusion) : {t_pso_np/t_fus:.1f}x')

## 5. Longstaff-Schwartz LSMC — American Option

In [ ]:
from models.longstaff import LSMC_Numpy, LSMC_CUDA

# CPU
ls_np  = LSMC_Numpy(mc, inverseType='benchmark_pinv')
price_ls_np, t_ls_np = ls_np.longstaff_schwartz_itm_path_fast()

# GPU – GJ (default)
ls_gpu = LSMC_CUDA(mc, preCalc=None, inverseType='GJ')
price_ls_gpu, t_ls_gpu = ls_gpu.longstaff_schwartz_itm_path_fast_hybrid()

# GPU – optimised
ls_gpu_opt = LSMC_CUDA(mc, preCalc='optimized')
price_ls_gpu_opt, t_ls_gpu_opt = ls_gpu_opt.longstaff_schwartz_itm_path_fast_hybrid()

print(f'\nSpeedup (LSMC-GJ)  : {t_ls_np/t_ls_gpu:.1f}x')
print(f'Speedup (LSMC-opt) : {t_ls_np/t_ls_gpu_opt:.1f}x')

## 6. Summary

In [ ]:
import pandas as pd

results = {
    'Method'  : ['MC-NumPy', 'MC-CUDA', 'PSO-NumPy', 'PSO-CUDA-hybrid', 'PSO-CUDA-scalar', 'PSO-CUDA-fusion', 'LSMC-NumPy', 'LSMC-CUDA-GJ', 'LSMC-CUDA-opt'],
    'Price'   : [price_np, price_gpu, price_pso_np, price_hyb, price_sc, price_fus, price_ls_np, price_ls_gpu, price_ls_gpu_opt],
    'Time(ms)': [t_np, t_gpu, t_pso_np, t_hyb, t_sc, t_fus, t_ls_np, t_ls_gpu, t_ls_gpu_opt],
}

df = pd.DataFrame(results)
df['Price'] = df['Price'].map('{:.4f}'.format)
df['Time(ms)'] = df['Time(ms)'].map('{:.2f}'.format)
print(df.to_string(index=False))

In [ ]:
# Free Z_d on device
mc.cleanUp()